<!-- paper-paired-role-banner -->
> 🟩 **정답 코드 · 오른쪽 창** — 왼쪽과 같은 셀 위치의 실행 가능한 참고 구현입니다. 복사하기 전에 구현 이유와 텐서 shape를 설명해 보세요.

# 17. ViT — 이미지를 patch token으로 — An Image is Worth 16x16 Words: Transformers for Image Recognition at Scale

- **원 논문:** [An Image is Worth 16x16 Words: Transformers for Image Recognition at Scale](https://arxiv.org/abs/2010.11929)
- **저자 / 발표:** Alexey Dosovitskiy et al. · ICLR (2021)
- **난이도 / 예상 시간:** 중급 · 약 75분
- **선수 지식:** convolution shape, self-attention, class token, positional embedding

이 노트북은 논문 결과 수치를 그대로 재현하는 대규모 benchmark가 아니라,
핵심 수식과 구조가 실제로 어떻게 동작하는지 확인하는 **하드웨어 인식 미니 재현**입니다.
PyTorch 학습은 CUDA → Apple MPS → CPU 순으로 자동 선택하며, 작은 NumPy·Python 실험은
전송 비용을 피하기 위해 CPU에서 실행될 수 있습니다.

## 재현 범위와 완료 기준

**이 노트북이 재현하는 것**

식 (1)의 patchify·linear projection·class token·position embedding과 식 (2)~(4)의 pre-LN Transformer encoder를 8×8 영상에 구현합니다. 위치 정보가 없으면 patch 순열을 구별하지 못하는 성질과 작은 막대 분류 학습을 확인합니다.

**원 논문과 다른 것**

JFT-300M/ImageNet 사전학습과 16×16 patch의 ViT-Base/Large/Huge는 생략합니다. 8×8 합성 영상, 2×2 patch, encoder 1층을 사용하므로 표 1~6의 전이 성능은 검증하지 않습니다.

완료하려면 TODO를 구현하고, 검증 셀의 `assert`를 모두 통과시키고, 마지막 관찰 질문에
자신의 말로 답하세요. 수치가 논문과 다르더라도 핵심 불변식과 비교 방향이 맞으면 성공입니다.

## 논문 ↔ 노트북 지도

        절 번호와 수식 번호는 위 링크의 대표 공개본을 기준으로 합니다. 판본에 따라 페이지는 달라질 수
        있으므로 **절 제목과 수식 번호**를 함께 사용하세요.

        | 원 논문의 위치 | 이 노트북의 대응 실습 | 확인할 증거 |
        |---|---|---|
        | §3.1 식 (1), 그림 1 | patchify·E projection·[class]·E_pos | [B,N+1,D] shape |
| §3.1 식 (2) | pre-LN multi-head attention residual | token shape 보존 |
| §3.1 식 (3) | pre-LN MLP residual | 유한 gradient |
| §3.1 식 (4) | class token 분류 head | 두 막대 class 정확도 |
| §3.1 Inductive bias | position embedding 제거 ablation | patch 순열 불변성 |

        > 읽기 순서: 먼저 해당 절을 훑고 → 코드를 예측해 작성하고 → 출력이 논문의 주장과 왜
        > 연결되는지 한 문장으로 설명합니다.

## 핵심 수식과 구현 설계

$$
z_0=[x_{class};x_p^1E;\dots;x_p^NE]+E_{pos},\qquad
N=\frac{HW}{P^2}
$$

- $x_p^i$는 $P\times P$ image patch, $E$는 patch projection, $E_{pos}$는 position embedding입니다.
- 2D image를 token sequence로 바꾸어 Transformer encoder에 입력하는 것이 핵심입니다.
- patchify, linear projection, class token, position embedding, encoder, head를 별도 모듈로 보입니다.
- image $[B,C,H,W]$는 patch $[B,N,CP^2]$를 거쳐 token $[B,N+1,D]$가 됩니다.

## 구현 선택 이유

- **원문에서 보존한 부분:** §3.1 식 (1), 그림 1
- **노트북 구현:** patchify·E projection·[class]·E_pos
- **이렇게 구현한 이유:** 완성된 고수준 model을 한 줄로 호출하면
  논문의 핵심 shape·mask·loss·상태 변화가 숨겨집니다. 따라서 이 계산을
  이름이 드러나는 class와 함수로 나누고 중간 tensor를 assertion으로
  검사합니다.
- **실패를 잡는 증거:** [B,N+1,D] shape를 출력 크기만 아니라
  구현 불변식으로 사용합니다.
- **축소 선택:** JFT-300M/ImageNet 사전학습과 16×16 patch의 ViT-Base/Large/Huge는 생략합니다. 8×8 합성 영상, 2×2 patch, encoder 1층을 사용하므로 표 1~6의 전이 성능은 검증하지 않습니다.는 기본 실행에서 생략합니다.
  이는 Windows CPU에서도 반복 실험이 가능하게 하면서 핵심 메커니즘은
  유지하기 위한 선택입니다.

### 구현 단계 1 · 실행 환경·데이터 계약 (setup 예외)

- **원문 위치:** 해당 없음 — import, seed, device, 로컬 데이터 로딩만 담당하며 논문의 학습 연산을 구현하지 않습니다.
- **Tensor shape 계약:** image [B,C,H,W] → patches [B,N,CP²] → tokens [B,N+1,D]
- **구현 이유:** 이후 셀의 비교가 재현 가능하도록 장치·dtype·seed와 입력 불변식을 먼저 고정합니다.
- **완료 증거:** 데이터 존재 여부, 입력 rank, 장치 선택 assertion과 환경 요약 출력이 통과해야 합니다.

In [ ]:
import math
import random

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import Tensor, nn
from torch.nn import functional as F
from llm_engineering_lab.acceleration import get_accelerator

random.seed(0)
np.random.seed(0)
torch.manual_seed(0)
ACCELERATOR = get_accelerator()
DEVICE = ACCELERATOR.device
if DEVICE.type == "cuda":
    torch.cuda.manual_seed_all(0)
print(ACCELERATOR.summary())
print("synthetic-data preprocessing stays on CPU; neural training uses DEVICE")

## 1) 식 (1)의 patch sequence

8×8 이미지를 겹치지 않는 2×2 patch로 나누면 `N=(8/2)^2=16`, patch vector 차원은
`P²C=4`입니다. `patchify`와 `unpatchify`가 정확한 역함수인지 먼저 확인합니다.

### 구현 단계 2 · 핵심 연산·모델

- **원문 위치:** §3.1 식 (1), 그림 1
- **이 셀의 책임:** patchify·E projection·[class]·E_pos
- **Tensor shape 계약:** image [B,C,H,W] → patches [B,N,CP²] → tokens [B,N+1,D]
- **구현 이유:** 수식을 숨긴 고수준 호출 대신 핵심 변환과 shape 경계를 직접 추적합니다.
- **완료 증거:** [B,N+1,D] shape. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
def patchify(x: Tensor, patch_size: int) -> Tensor:
    b, c, h, w = x.shape
    if h % patch_size or w % patch_size:
        raise ValueError("height and width must be divisible by patch_size")
    x = x.reshape(b, c, h // patch_size, patch_size, w // patch_size, patch_size)
    return x.permute(0, 2, 4, 3, 5, 1).reshape(b, -1, patch_size * patch_size * c)


def unpatchify(tokens: Tensor, patch_size: int, height: int, width: int) -> Tensor:
    b, _, flat = tokens.shape
    c = flat // (patch_size * patch_size)
    x = tokens.reshape(
        b,
        height // patch_size,
        width // patch_size,
        patch_size,
        patch_size,
        c,
    )
    return x.permute(0, 5, 1, 3, 2, 4).reshape(b, c, height, width)


checker = torch.arange(2 * 1 * 8 * 8, dtype=torch.float32).reshape(2, 1, 8, 8)
patches = patchify(checker, 2)
assert patches.shape == (2, 16, 4)
assert torch.equal(unpatchify(patches, 2, 8, 8), checker)
print("patch tokens:", tuple(patches.shape))

## 2) 식 (1)~(4)의 TinyViT

원 논문은 각 encoder block을 `x' = MSA(LN(x)) + x`,
`x = MLP(LN(x')) + x'`로 씁니다. 여기서는 이 pre-LN 순서를 고수준
`TransformerEncoder` 호출에 숨기지 않습니다. `ViTSelfAttention`은 Q/K/V와 head
결합을, `ViTMLP`는 GELU를 포함한 식 (3)을, `ViTEncoderBlock`은 두 residual
경로를 각각 드러냅니다.

### 구현 단계 3 · 핵심 연산·모델

- **원문 위치:** §3.1 식 (1), 그림 1 / §3.1 식 (2) / §3.1 식 (3) / §3.1 식 (4)
- **이 셀의 책임:** patchify·E projection·[class]·E_pos / pre-LN multi-head attention residual / pre-LN MLP residual / class token 분류 head
- **Tensor shape 계약:** image [B,C,H,W] → patches [B,N,CP²] → tokens [B,N+1,D]
- **구현 이유:** 수식을 숨긴 고수준 호출 대신 핵심 변환과 shape 경계를 직접 추적합니다.
- **완료 증거:** [B,N+1,D] shape / token shape 보존 / 유한 gradient / 두 막대 class 정확도. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
class ViTSelfAttention(nn.Module):
    def __init__(self, dim: int, heads: int):
        super().__init__()
        if dim % heads:
            raise ValueError("dim must be divisible by heads")
        self.heads = heads
        self.head_dim = dim // heads
        self.query_key_value = nn.Linear(dim, 3 * dim, bias=False)
        self.output = nn.Linear(dim, dim, bias=False)

    def _split_heads(self, tensor: Tensor) -> Tensor:
        batch, tokens, width = tensor.shape
        split = tensor.view(
            batch,
            tokens,
            self.heads,
            width // self.heads,
        )
        return split.transpose(1, 2)

    def _merge_heads(self, tensor: Tensor) -> Tensor:
        batch, _, tokens, _ = tensor.shape
        merged = tensor.transpose(1, 2).contiguous()
        return merged.view(
            batch,
            tokens,
            self.heads * self.head_dim,
        )

    def forward(self, hidden: Tensor) -> tuple[Tensor, Tensor]:
        query, key, value = self.query_key_value(hidden).chunk(3, dim=-1)
        query = self._split_heads(query)
        key = self._split_heads(key)
        value = self._split_heads(value)
        scores = query @ key.transpose(-2, -1)
        scores = scores / math.sqrt(self.head_dim)
        weights = torch.softmax(scores, dim=-1)
        context = self._merge_heads(weights @ value)
        return self.output(context), weights


class ViTMLP(nn.Module):
    def __init__(self, dim: int, hidden_dim: int):
        super().__init__()
        self.input = nn.Linear(dim, hidden_dim)
        self.output = nn.Linear(hidden_dim, dim)

    def forward(self, hidden: Tensor) -> Tensor:
        return self.output(F.gelu(self.input(hidden)))


class ViTEncoderBlock(nn.Module):
    def __init__(self, dim: int, heads: int, mlp_dim: int):
        super().__init__()
        self.attention_norm = nn.LayerNorm(dim)
        self.attention = ViTSelfAttention(dim, heads)
        self.mlp_norm = nn.LayerNorm(dim)
        self.mlp = ViTMLP(dim, mlp_dim)

    def forward(self, hidden: Tensor) -> tuple[Tensor, Tensor]:
        attended, weights = self.attention(self.attention_norm(hidden))
        hidden = hidden + attended
        hidden = hidden + self.mlp(self.mlp_norm(hidden))
        return hidden, weights


class TinyViT(nn.Module):
    def __init__(self, image_size=8, patch_size=2, dim=24, heads=3, classes=2):
        super().__init__()
        self.patch_size = patch_size
        count = (image_size // patch_size) ** 2
        self.projection = nn.Linear(patch_size * patch_size, dim)
        self.cls_token = nn.Parameter(torch.zeros(1, 1, dim))
        self.position = nn.Parameter(torch.randn(1, count + 1, dim) * 0.02)
        self.encoder = ViTEncoderBlock(dim, heads, mlp_dim=2 * dim)
        self.norm = nn.LayerNorm(dim)
        self.head = nn.Linear(dim, classes)

    def embed_tokens(self, x: Tensor, use_position: bool = True) -> Tensor:
        patch_tokens = self.projection(patchify(x, self.patch_size))
        cls = self.cls_token.expand(x.size(0), -1, -1)
        tokens = torch.cat([cls, patch_tokens], dim=1)
        return tokens + self.position if use_position else tokens

    def encode_tokens(self, tokens: Tensor) -> Tensor:
        encoded, _ = self.encoder(tokens)
        return self.norm(encoded[:, 0])

    def forward(self, x: Tensor) -> Tensor:
        return self.head(self.encode_tokens(self.embed_tokens(x)))


vit = TinyViT()
dummy = torch.randn(3, 1, 8, 8)
assert vit.embed_tokens(dummy).shape == (3, 17, 24)
assert vit(dummy).shape == (3, 2)
embedded = vit.embed_tokens(dummy)
encoded, attention_weights = vit.encoder(embedded)
assert encoded.shape == embedded.shape
assert attention_weights.shape == (3, 3, 17, 17)
assert torch.allclose(
    attention_weights.sum(dim=-1),
    attention_weights.new_ones(3, 3, 17),
    atol=1e-6,
)

### 구현 단계 4 · 논문 주장 확인 실험

- **원문 위치:** §3.1 Inductive bias
- **이 셀의 책임:** position embedding 제거 ablation
- **Tensor shape 계약:** image [B,C,H,W] → patches [B,N,CP²] → tokens [B,N+1,D]
- **구현 이유:** 한 변수만 바꾸어 해당 메커니즘이 출력에 미치는 영향을 분리합니다.
- **완료 증거:** patch 순열 불변성. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
tokens_no_pos = vit.embed_tokens(dummy, use_position=False)
permutation = torch.randperm(16)
permuted_no_pos = torch.cat(
    [tokens_no_pos[:, :1], tokens_no_pos[:, 1:][:, permutation]],
    dim=1,
)
no_pos_a = vit.encode_tokens(tokens_no_pos)
no_pos_b = vit.encode_tokens(permuted_no_pos)

tokens_pos = tokens_no_pos + vit.position
permuted_content_same_positions = torch.cat(
    [tokens_no_pos[:, :1], tokens_no_pos[:, 1:][:, permutation]],
    dim=1,
)
permuted_content_same_positions = permuted_content_same_positions + vit.position
pos_a = vit.encode_tokens(tokens_pos)
pos_b = vit.encode_tokens(permuted_content_same_positions)
assert torch.allclose(no_pos_a, no_pos_b, atol=1e-5)
assert not torch.allclose(pos_a, pos_b, atol=1e-5)
print("no-pos permutation delta:", float((no_pos_a - no_pos_b).abs().max()))
print("with-pos permutation delta:", float((pos_a - pos_b).abs().max()))

### 구현 단계 5 · 목적함수·학습 update

- **원문 위치:** §3.1 식 (4)
- **이 셀의 책임:** class token 분류 head
- **Tensor shape 계약:** image [B,C,H,W] → patches [B,N,CP²] → tokens [B,N+1,D]
- **구현 이유:** forward 결과와 논문 objective, gradient update의 책임을 분리합니다.
- **완료 증거:** 두 막대 class 정확도. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
generator = torch.Generator().manual_seed(17)
train_x = 0.08 * torch.randn(160, 1, 8, 8, generator=generator)
train_y = torch.arange(160) % 2
for index, label in enumerate(train_y.tolist()):
    if label == 0:
        train_x[index, :, :, 3:5] += 1.0
    else:
        train_x[index, :, 3:5, :] += 1.0
train_x = train_x.clamp(0, 1)

vit = ACCELERATOR.move(TinyViT())
train_x_device, train_y_device = ACCELERATOR.move(train_x, train_y)
opt = torch.optim.Adam(vit.parameters(), lr=0.01)
history = []
for _ in range(70):
    loss = F.cross_entropy(vit(train_x_device), train_y_device)
    opt.zero_grad()
    loss.backward()
    opt.step()
    history.append(float(loss.detach().cpu()))
accuracy = (vit(train_x_device).argmax(1) == train_y_device).float().mean().item()
assert history[-1] < history[0] and accuracy > 0.95
print(f"training accuracy={accuracy:.1%}, loss={history[-1]:.4f}")
plt.figure(figsize=(5, 2.5))
plt.plot(history)
plt.title("TinyViT mini training")
plt.xlabel("step")
plt.ylabel("cross entropy")
plt.grid(alpha=0.2)
plt.show()

## 마무리 해설과 다음 실험

1. 핵심 연산과 구현 위치를 원 논문의 용어로 다시 연결해 설명하세요.
2. 이 미니 실험은 `JFT-300M/ImageNet 사전학습과 16×16 patch의 ViT-Base/Large/Huge는 생략합니다. 8×8 합성 영상, 2×2 patch, encoder 1층을 사용하므로 표 1~6의 전이 성능은 검증하지 않습니다.`를 재현하지 않으므로 논문의 최종 성능을 검증한 것이 아닙니다.
3. 다음에는 데이터 크기, 모델 폭, 학습 횟수 중 **하나만** 바꾸고 같은 seed에서 metric을 비교하세요.

정답의 수치 자체보다, shape·확률 정규화·mask·gradient 같은 불변식을 먼저 확인하는 습관이
논문 구현에서 가장 강한 디버깅 도구입니다.